# Person 3 charts: 3, 6 and 9
Questions: **Q5** (Time) and **Q8** (Language and geography).

Reads from `netflix_analysis.db`. Run `python src/run_pipeline.py` first so the database exists.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Find the database (works from the notebooks/ folder or the main folder)
DB = next(p for p in [Path("../netflix_analysis.db"), Path("netflix_analysis.db")] if p.exists())
con = sqlite3.connect(DB)

# Use Person 4's shared colours if palette.py exists, else use these.
try:
    import sys; sys.path.append("../src"); sys.path.append("../app"); sys.path.append(".")
    from palette import RED, DARK, GREY
except Exception:
    RED, DARK, GREY = "#E50914", "#221F1F", "#B3B3B3"

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "figure.dpi": 110})

## Chart 3: Release-year trend (line graph)

In [ ]:
df = pd.read_sql("""
    SELECT release_year, media_type, COUNT(*) AS n
    FROM titles
    WHERE release_year IS NOT NULL
    GROUP BY release_year, media_type
    ORDER BY release_year
""", con)
pivot = df.pivot(index="release_year", columns="media_type", values="n").fillna(0)

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = {"movie": RED, "tv": DARK}
for col in pivot.columns:
    ax.plot(pivot.index, pivot[col], color=colors.get(col, GREY), linewidth=2,
            label="Movies" if col == "movie" else "TV series")
ax.set_title("Number of titles by release year")
ax.set_xlabel("Release year"); ax.set_ylabel("Number of titles")
ax.legend(frameon=False)
fig.text(0.01, -0.02, "Source: netflix_analysis.db (titles table)", fontsize=8, color="gray")
plt.tight_layout(); plt.savefig("chart3_release_year_trend.png", bbox_inches="tight"); plt.show()

**Insight (Q5):** _Write 1 or 2 sentences here after you look at the chart. Example: which years have the most titles, and is most of the catalog recent?_

## Chart 6: Top 10 producing countries (bar chart)

In [ ]:
top = pd.read_sql("""
    SELECT country, COUNT(*) AS n
    FROM countries
    GROUP BY country
    ORDER BY n DESC
    LIMIT 10
""", con).sort_values("n")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(top["country"], top["n"], color=RED)
for y, v in enumerate(top["n"]):
    ax.text(v, y, f" {v}", va="center", fontsize=9)
ax.set_title("Top 10 producing countries")
ax.set_xlabel("Number of titles"); ax.set_ylabel("")
fig.text(0.01, -0.02, "Source: netflix_analysis.db (countries table)", fontsize=8, color="gray")
plt.tight_layout(); plt.savefig("chart6_top_countries.png", bbox_inches="tight"); plt.show()

**Insight (Q8):** _Write 1 or 2 sentences here. Example: which country leads and by how much?_

## Chart 9: Content added per year (trend line)
Only the Kaggle data has `date_added`. TMDB rows have no value here, so this chart uses Kaggle rows only.

In [ ]:
added = pd.read_sql("""
    SELECT CAST(strftime('%Y', date_added) AS INTEGER) AS year_added, COUNT(*) AS n
    FROM titles
    WHERE date_added IS NOT NULL
    GROUP BY year_added
    ORDER BY year_added
""", con)

if added.empty:
    print("No date_added values in the database yet. Load the Kaggle data to draw this chart.")
else:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(added["year_added"], added["n"], color=RED, linewidth=2.2, marker="o")
    ax.set_title("Titles added to Netflix per year")
    ax.set_xlabel("Year added"); ax.set_ylabel("Number of titles")
    fig.text(0.01, -0.02, "Source: Kaggle dataset (date_added field)", fontsize=8, color="gray")
    plt.tight_layout(); plt.savefig("chart9_content_added.png", bbox_inches="tight"); plt.show()

**Insight (Q5):** _Write 1 or 2 sentences here. Example: when did additions peak, and did they drop after?_